# Lesson 05 — Practical Applications: Denoising and Sharpening in Frequency Domain

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)

# Add periodic noise (horizontal stripes — a real camera problem)
noisy = gray.copy()
for i in range(0, gray.shape[0], 10):
    noisy[i, :] += 50
noisy = np.clip(noisy, 0, 255)

dft         = cv2.dft(noisy, flags=cv2.DFT_COMPLEX_OUTPUT)
dft_shifted = np.fft.fftshift(dft)
magnitude   = cv2.magnitude(dft_shifted[:,:,0], dft_shifted[:,:,1])
mag_log     = np.log1p(magnitude)

h, w = gray.shape
cx, cy = w//2, h//2

# The stripe noise appears as bright dots along the vertical axis of spectrum
# Suppress them with a notch filter (zero out those specific frequencies)
mask = np.ones((h, w, 2), np.float32)
# Zero out the horizontal stripe frequencies (vertical axis, away from center)
for dy in range(5, h//2):
    mask[cy-dy, cx-2:cx+3] = 0
    mask[cy+dy, cx-2:cx+3] = 0

filtered  = dft_shifted * mask
back      = np.fft.ifftshift(filtered)
destriped = cv2.idft(back, flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
destriped = cv2.normalize(destriped, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, im, t in zip(axes,
    [noisy.astype(np.uint8), mag_log, destriped],
    ['Striped noisy image', 'Magnitude spectrum\n(bright dots = stripe freq)', 'Destriped result']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.show()

## Key Takeaway
Periodic noise (scanning lines, power line interference, moire patterns) appears as isolated bright
spots in the frequency spectrum. Zero them out to remove the pattern cleanly.